### Diccionario de datos ChileCompra - Órdenes de Compra

| Campo                | Descripción |
|----------------------|-------------|
| `codigoOC`           | Código único de la orden de compra. Suele estar en el formato XX-YY-ZZ. |
| `NroLicitacion`      | Número de licitación asociado a la orden de compra. Si es trato directo, puede venir vacío. |
| `FechaEnvioOC`       | Fecha en que se envió la orden de compra. Formato típico: YYYY-MM-DD. |
| `NombreOC`           | Nombre o título de la orden de compra. Describe brevemente el contenido. |
| `EstadoOC`           | Estado actual de la OC: Aceptada, En Proceso, Anulada, etc. |
| `ProcedenciaOC`      | Forma en que se originó la OC: puede ser Licitación Pública, Trato Directo, Convenio Marco, etc. |
| `MontoTotalOC`       | Monto total de la OC (con IVA incluido). Generalmente expresado en CLP. |
| `MontoNetoOC_CLP`    | Monto neto (sin IVA) en pesos chilenos. |
| `UnidadCompra`       | Nombre de la unidad compradora (departamento o división dentro de la institución). |
| `RegionUnidadCompra` | Región geográfica de la unidad compradora|
| `Institucion`        | Institución pública que realiza la compra (ej. Ministerio de Salud). |
| `Sector`             | Sector al que pertenece la institución (ej. Salud, Educación, etc.). |
| `Proveedor`          | Nombre del proveedor que recibe la orden de compra. |
| `ProveedorRUT`       | Rut Único Tributario del proveedor (formato: 12.345.678-9). |
| `TamanoProveedor`    | Tamaño de la empresa: Micro, Pequeña, Mediana, Grande. |
| `RegionProveedor`    | Región donde está ubicado el proveedor. |
| `RubroN1, N2, N3`     | Ni idea |
| `ONUProducto`        | Código del producto o servicio según clasificación de la ONU (UNSPSC). Estandariza los productos a nivel internacional. |
| `NombreItem`         | Nombre del ítem o servicio específico comprado. |
| `CantidadItem`       | Cantidad de unidades del ítem comprado. |
| `MontoTotalItem`     | Monto total del ítem (Cantidad * Precio unitario, con IVA). |
| `Modalidad`          | Modalidad de compra: Licitación Pública, Convenio Marco, Trato Directo, Licitación Privada, etc. |


In [105]:
import plotly  
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [106]:
data= pd.read_csv('data/data.csv')
unspsc_data = pd.read_csv('data/clean_unspsc_data.csv')

/var/folders/rn/8kg7t05x7l3fy1mwbgcdkw240000gp/T/ipykernel_7363/779305669.py:1: DtypeWarning:

Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.



In [107]:
# Drop not used columns 'codigoOC', 'NroLicitacion', 'NombreOC', 'ProveedorRUT', 'RubroN1', 'RubroN2', 'RubroN3'
df = data.drop(columns=['codigoOC', 'NroLicitacion',  'RubroN1', 'RubroN2', 'RubroN3', 'Institucion', 'Financiamiento', 'UnidadCompra'])

In [108]:
# Hacer merge entre los DataFrames
data_merged = pd.merge(
    df, 
    unspsc_data, 
    left_on='CodigoProductoONU', 
    right_on='Commodity Code', 
    how='left' 
)

data_merged = data_merged.drop(columns=['Commodity Code'])
# Ver las primeras filas para verificar
data_merged.head()

,NombreOC,EstadoOC,RegionUnidadCompra,Proveedor,CodigoProductoONU,NombreItem,DescripcionItem,CantidadItem,MontoTotalItem,TipoCompra,Precio_Unitario_TOTAL,Segment Code,Segment Name,Family Code,Family Name,Class Code,Class Name,Commodity Name
0,ALIMENTOS SENDA PREVIENE,Aceptada,Region de los Lagos,LUIS HUMBERTO CASAS NEGRON,50202304,JUGOS Y NÉCTAR ENVASADOS,JUGO DE 200 CC,100,33600,Licitacion,336.0,50000000.0,Food Beverage and Tobacco Products,50200000.0,Beverages,50202300.0,Non alcoholic beverages,Shelf stable juice
1,INSUMOS ALIMENTICIOS L.B.N.M,Aceptada,Region del Biobio,SILVIA SANDOVAL CERDA,93131607,SERVICIOS DE DISTRIBUCIÓN DE ALIMENTOS,NaN,1,60000,Licitacion,60000.0,93000000.0,Politics and Civic Affairs Services,93130000.0,Humanitarian aid and relief,93131600.0,Food and nutrition policy planning and programs,Food distribution services
2,INSUMOS ALIMENTICIOS L.B.N.M,Aceptada,Region del Biobio,SILVIA SANDOVAL CERDA,93131607,SERVICIOS DE DISTRIBUCIÓN DE ALIMENTOS,NaN,3,7500,Licitacion,2500.0,93000000.0,Politics and Civic Affairs Services,93130000.0,Humanitarian aid and relief,93131600.0,Food and nutrition policy planning and programs,Food distribution services
3,INSUMOS ALIMENTICIOS L.B.N.M,Aceptada,Region del Biobio,SILVIA SANDOVAL CERDA,93131607,SERVICIOS DE DISTRIBUCIÓN DE ALIMENTOS,NaN,400,60000,Licitacion,150.0,93000000.0,Politics and Civic Affairs Services,93130000.0,Humanitarian aid and relief,93131600.0,Food and nutrition policy planning and programs,Food distribution services
4,ORDEN DE COMPRA DESDE 2446-143-LE25,Recepcion Conforme,Region de Coquimbo,ESCENCIAL SPA,14111531,LIBROS O CUADERNOS DE REGISTRO,CUADERNO UNIVERSITARIO DOBLE ESPIRAL 100 HJS.,10,8700,Licitacion,870.0,14000000.0,Paper Materials and Products,14110000.0,Paper products,14111500.0,Printing and writing paper,Log books or pads


In [112]:
# variables de control
MINIMO_TRANSACCIONES = 5
MINIMO_SOBREPRECIO_CLP_RELEVANTE = 10000
FACTOR_IQR = 2.0

#  Calcular límites y contadores 
p25_item = data_merged.groupby('Commodity Name')['Precio_Unitario_TOTAL'].transform('quantile', 0.25)
p75_item = data_merged.groupby('Commodity Name')['Precio_Unitario_TOTAL'].transform('quantile', 0.75)
iqr_item = p75_item - p25_item
n_transacciones = data_merged.groupby('Commodity Name')['Precio_Unitario_TOTAL'].transform('count')

# límite estadístico
limite_sobreprecio_stats = p75_item + (FACTOR_IQR * iqr_item)

# diferencia monetaria
diferencia_monetaria = data_merged['Precio_Unitario_TOTAL'] - p75_item

nombre_columna_cantidad = 'CantidadItem'  

data_merged['Monto_Total_Orden'] = data_merged['Precio_Unitario_TOTAL'] * data_merged[nombre_columna_cantidad]


# --- Etiquetar con todas las condiciones ---

#  Empezamos asumiendo (False)
data_merged['is_expensive'] = False

# Definimos las condiciones para un precio expensivo significativo
condicion_estadistica = (data_merged['Precio_Unitario_TOTAL'] > limite_sobreprecio_stats)
condicion_suficientes_datos = (n_transacciones >= MINIMO_TRANSACCIONES)
condicion_monetaria_relevante = (diferencia_monetaria >= MINIMO_SOBREPRECIO_CLP_RELEVANTE)

# El monto total de la línea debe ser > 200.000 y otro, lo voy a dejar en ese por ahora
condicion_monto_total_relevante = (data_merged['Monto_Total_Orden'] > 200000)


# Aplicamos la etiqueta 
data_merged.loc[
    condicion_estadistica & 
    condicion_suficientes_datos & 
    condicion_monetaria_relevante &
    condicion_monto_total_relevante,  
    'is_expensive'
] = True

# 3. Anulamos los que no tienen datos suficientes
condicion_insuficiente = (n_transacciones < MINIMO_TRANSACCIONES)
data_merged.loc[condicion_insuficiente, 'is_expensive'] = False



In [110]:
data_merged

,NombreOC,EstadoOC,RegionUnidadCompra,Proveedor,CodigoProductoONU,NombreItem,DescripcionItem,CantidadItem,MontoTotalItem,TipoCompra,Precio_Unitario_TOTAL,Segment Code,Segment Name,Family Code,Family Name,Class Code,Class Name,Commodity Name,Monto_Total_Orden,is_expensive
0,ALIMENTOS SENDA PREVIENE,Aceptada,Region de los Lagos,LUIS HUMBERTO CASAS NEGRON,50202304,JUGOS Y NÉCTAR ENVASADOS,JUGO DE 200 CC,100,33600,Licitacion,336.0,50000000.0,Food Beverage and Tobacco Products,50200000.0,Beverages,50202300.0,Non alcoholic beverages,Shelf stable juice,33600.0,False
1,INSUMOS ALIMENTICIOS L.B.N.M,Aceptada,Region del Biobio,SILVIA SANDOVAL CERDA,93131607,SERVICIOS DE DISTRIBUCIÓN DE ALIMENTOS,NaN,1,60000,Licitacion,60000.0,93000000.0,Politics and Civic Affairs Services,93130000.0,Humanitarian aid and relief,93131600.0,Food and nutrition policy planning and programs,Food distribution services,60000.0,False
2,INSUMOS ALIMENTICIOS L.B.N.M,Aceptada,Region del Biobio,SILVIA SANDOVAL CERDA,93131607,SERVICIOS DE DISTRIBUCIÓN DE ALIMENTOS,NaN,3,7500,Licitacion,2500.0,93000000.0,Politics and Civic Affairs Services,93130000.0,Humanitarian aid and relief,93131600.0,Food and nutrition policy planning and programs,Food distribution services,7500.0,False
3,INSUMOS ALIMENTICIOS L.B.N.M,Aceptada,Region del Biobio,SILVIA SANDOVAL CERDA,93131607,SERVICIOS DE DISTRIBUCIÓN DE ALIMENTOS,NaN,400,60000,Licitacion,150.0,93000000.0,Politics and Civic Affairs Services,93130000.0,Humanitarian aid and relief,93131600.0,Food and nutrition policy planning and programs,Food distribution services,60000.0,False
4,ORDEN DE COMPRA DESDE 2446-143-LE25,Recepcion Conforme,Region de Coquimbo,ESCENCIAL SPA,14111531,LIBROS O CUADERNOS DE REGISTRO,CUADERNO UNIVERSITARIO DOBLE ESPIRAL 100 HJS.,10,8700,Licitacion,870.0,14000000.0,Paper Materials and Products,14110000.0,Paper products,14111500.0,Printing and writing paper,Log books or pads,8700.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747857,INSUMOS PARA TALLERES- DIDECO 1045-1046,Recepcion Conforme,Region de Valparaiso,KARINA PAIS LORCA,50181905,GALLETAS DULCES O PASTELITOS,BOLSAS DE TABLETON GALLETAS BAÑADAS EN CHOCOLA...,4,13172,Compra Agil,3293.0,50000000.0,Food Beverage and Tobacco Products,50180000.0,Bread and bakery products,50181900.0,Bread and biscuits and cookies,Sweet biscuits or cookies,13172.0,False
747858,INSUMOS PARA TALLERES- DIDECO 1045-1046,Recepcion Conforme,Region de Valparaiso,KARINA PAIS LORCA,50181905,GALLETAS DULCES O PASTELITOS,PAQUETES GALLETA TRITON SABORES SURTIDOS 126 GRS,12,12360,Compra Agil,1030.0,50000000.0,Food Beverage and Tobacco Products,50180000.0,Bread and bakery products,50181900.0,Bread and biscuits and cookies,Sweet biscuits or cookies,12360.0,False
747859,MATERIALES DIDACTICOS- OLN DIDECO,Recepcion Conforme,Region de Valparaiso,SOCIEDAD COMERCIAL VERCON SOCIEDAD POR ACCIONE...,44111804,PAPELES DE DIBUJO,RESMA OPALINA LISA BLANCA 100 HOJAS CARTA,3,28500,Compra Agil,9500.0,44000000.0,Office Equipment and Accessories and Supplies,44110000.0,Office and desk accessories,44111800.0,Drafting supplies,Drafting papers,28500.0,False
747860,MATERIALES DIDACTICOS- OLN DIDECO,Recepcion Conforme,Region de Valparaiso,SOCIEDAD COMERCIAL VERCON SOCIEDAD POR ACCIONE...,44121612,CUCHILLOS CARTONEROS,CUCHILLO CARTONERO GRANDE CON BLOQUEO,4,5600,Compra Agil,1400.0,44000000.0,Office Equipment and Accessories and Supplies,44120000.0,Office supplies,44121600.0,Desk supplies,Paper cutters or refills,5600.0,False


In [113]:

#Calcular el total de productos caros
total_expensive = data_merged['is_expensive'].sum()

# Contar productos caros por Segment Name
expensive_by_segment = data_merged[data_merged['is_expensive'] == True].groupby('Segment Name').size()

# Calcular porcentajes
percentage_by_segment = (expensive_by_segment / total_expensive * 100).sort_values(ascending=False)

# Mostrar top 10
print("TOP 10 SEGMENTOS CON MAYOR CONCENTRACIÓN DE PRODUCTOS CAROS")
print("=" * 70)
print(f"Total de productos caros: {total_expensive:,}")
print("=" * 70)

for i, (segment, percentage) in enumerate(percentage_by_segment.head(10).items(), 1):
    count = expensive_by_segment[segment]
    print(f"{i:2d}. {segment[:50]:<50} = {percentage:5.1f}% ({count:,} productos)")

print("=" * 70)

# También mostrar estadísticas adicionales
print(f"\nLos top 10 segmentos concentran: {percentage_by_segment.head(10).sum():.1f}% del total")

TOP 10 SEGMENTOS CON MAYOR CONCENTRACIÓN DE PRODUCTOS CAROS
Total de productos caros: 61,692
 1. Food Beverage and Tobacco Products                 =   7.6% (4,692 productos)
 2. Office Equipment and Accessories and Supplies      =   6.7% (4,131 productos)
 3. Manufacturing Components and Supplies              =   5.6% (3,484 productos)
 4. Musical Instruments and Games and Toys and Arts an =   5.3% (3,254 productos)
 5. Medical Equipment and Accessories and Supplies     =   5.0% (3,078 productos)
 6. Paper Materials and Products                       =   4.8% (2,980 productos)
 7. Transportation and Storage and Mail Services       =   3.9% (2,426 productos)
 8. Structures and Building and Construction and Manuf =   3.9% (2,401 productos)
 9. Cleaning Equipment and Supplies                    =   3.5% (2,157 productos)
10. Management and Business Professionals and Administ =   3.2% (1,953 productos)

Los top 10 segmentos concentran: 49.5% del total


In [115]:
# Calcular el total de productos caros
total_expensive = data_merged['is_expensive'].sum()

# Contar productos caros por RegionUnidadCompra
expensive_by_region = data_merged[data_merged['is_expensive'] == True].groupby('RegionUnidadCompra').size()

# Contar total de compras por región
total_by_region = data_merged.groupby('RegionUnidadCompra').size()

# Calcular porcentaje de compras caras por región
percentage_expensive_by_region = (expensive_by_region / total_by_region * 100).sort_values(ascending=False)

# Filtrar regiones que tienen al menos una compra cara
percentage_expensive_by_region = percentage_expensive_by_region.dropna()

# Mostrar todas las regiones
print("PORCENTAJE DE COMPRAS CON SOBREPRECIO POR REGIÓN")
print("=" * 80)
print(f"Total de productos caros a nivel nacional: {total_expensive:,}")
print("=" * 80)

for i, (region, percentage) in enumerate(percentage_expensive_by_region.items(), 1):
    expensive_count = expensive_by_region[region]
    total_count = total_by_region[region]
    print(f"{i:2d}. {region:<35} = {percentage:5.1f}% ({expensive_count:,} de {total_count:,} compras)")

print("=" * 80)

# Estadísticas adicionales
print(f"\nPromedio nacional de compras con sobreprecio: {(total_expensive / len(data_merged) * 100):.1f}%")
print(f"Número de regiones con compras caras: {len(percentage_expensive_by_region)}")

PORCENTAJE DE COMPRAS CON SOBREPRECIO POR REGIÓN
Total de productos caros a nivel nacional: 61,692
 1. Region de los Lagos                 =  11.1% (7,711 de 69,410 compras)
 2. Region del Maule                    =  11.0% (7,386 de 67,204 compras)
 3. Region de la Araucania              =  10.5% (7,142 de 68,048 compras)
 4. Region Aysen del General Carlos IbaNez del Campo =   9.7% (825 de 8,474 compras)
 5. Region del Libertador General Bernardo OHiggins =   8.7% (4,823 de 55,515 compras)
 6. Region Metropolitana de Santiago    =   8.6% (8,066 de 93,560 compras)
 7. Region de Valparaiso                =   7.8% (6,468 de 82,640 compras)
 8. Region de Coquimbo                  =   7.7% (3,147 de 40,913 compras)
 9. Region del Biobio                   =   7.5% (6,860 de 91,116 compras)
10. Region de Arica y Parinacota        =   7.3% (458 de 6,298 compras)
11. Region de Los Rios                  =   5.9% (2,198 de 37,249 compras)
12. Region de Atacama                   =   5.6% (1,019 d

In [118]:
# Calcular el gasto total en sobreprecios por Segment Name
expensive_spending_by_segment = data_merged[data_merged['is_expensive'] == True].groupby('Segment Name')['Monto_Total_Orden'].sum()

# Ordenar por mayor gasto
expensive_spending_by_segment = expensive_spending_by_segment.sort_values(ascending=False)

# Calcular el total gastado en sobreprecios
total_overspend = expensive_spending_by_segment.sum()

# Calcular porcentajes del total
percentage_spending_by_segment = (expensive_spending_by_segment / total_overspend * 100)

# Mostrar top 10
print("TOP 10 SEGMENTOS CON MAYOR GASTO EN SOBREPRECIOS")
print("=" * 80)
print(f"Total gastado en sobreprecios: ${total_overspend:,.0f} CLP")
print("=" * 80)

for i, (segment, amount) in enumerate(expensive_spending_by_segment.head(10).items(), 1):
    percentage = percentage_spending_by_segment[segment]
    # Contar cuántas compras caras hay en este segmento
    expensive_count = data_merged[(data_merged['is_expensive'] == True) & 
                                 (data_merged['Segment Name'] == segment)].shape[0]
    
    print(f"{i:2d}. {segment[:55]:<55}")
    print(f"    Gasto en sobreprecios: ${amount:,.0f} CLP ({percentage:5.1f}%)")
    print(f"    Número de compras caras: {expensive_count:,}")
    print()

print("=" * 80)

# Estadísticas adicionales
top_10_spending = expensive_spending_by_segment.head(10).sum()
print(f"Los top 10 segmentos concentran: ${top_10_spending:,.0f} CLP ({top_10_spending/total_overspend*100:.1f}%) del gasto total en sobreprecios")

TOP 10 SEGMENTOS CON MAYOR GASTO EN SOBREPRECIOS
Total gastado en sobreprecios: $59,495,311,114,801,340,416 CLP
 1. Food Beverage and Tobacco Products                     
    Gasto en sobreprecios: $8,344,748,465,475,142,656 CLP ( 14.0%)
    Número de compras caras: 4,692

 2. Fuels and Fuel Additives and Lubricants and Anti corros
    Gasto en sobreprecios: $7,419,588,727,603,954,688 CLP ( 12.5%)
    Número de compras caras: 1,100

 3. Office Equipment and Accessories and Supplies          
    Gasto en sobreprecios: $4,528,607,569,389,454,336 CLP (  7.6%)
    Número de compras caras: 4,131

 4. Manufacturing Components and Supplies                  
    Gasto en sobreprecios: $3,985,145,543,758,541,824 CLP (  6.7%)
    Número de compras caras: 3,484

 5. Paper Materials and Products                           
    Gasto en sobreprecios: $3,108,629,228,745,925,120 CLP (  5.2%)
    Número de compras caras: 2,980

 6. Structures and Building and Construction and Manufactur
    Gasto en 

In [119]:
# Calcular el gasto total en sobreprecios por Commodity Name
expensive_spending_by_commodity = data_merged[data_merged['is_expensive'] == True].groupby('Commodity Name')['Monto_Total_Orden'].sum()

# Ordenar por mayor gasto
expensive_spending_by_commodity = expensive_spending_by_commodity.sort_values(ascending=False)

# Calcular el total gastado en sobreprecios
total_overspend = expensive_spending_by_commodity.sum()

# Para cada commodity, obtener estadísticas de precios
commodity_stats = []

for commodity in expensive_spending_by_commodity.head(10).index:
    # Filtrar datos de esta commodity
    commodity_data = data_merged[data_merged['Commodity Name'] == commodity]
    expensive_data = commodity_data[commodity_data['is_expensive'] == True]
    
    # Estadísticas de precios
    precio_promedio_expensive = expensive_data['Precio_Unitario_TOTAL'].mean()
    precio_p75_esperado = commodity_data['Precio_Unitario_TOTAL'].quantile(0.75)
    precio_promedio_normal = commodity_data['Precio_Unitario_TOTAL'].mean()
    
    # Calcular porcentaje de sobreprecio
    sobreprecio_vs_p75 = ((precio_promedio_expensive - precio_p75_esperado) / precio_p75_esperado * 100)
    sobreprecio_vs_promedio = ((precio_promedio_expensive - precio_promedio_normal) / precio_promedio_normal * 100)
    
    commodity_stats.append({
        'commodity': commodity,
        'gasto_total': expensive_spending_by_commodity[commodity],
        'precio_avg_expensive': precio_promedio_expensive,
        'precio_p75_esperado': precio_p75_esperado,
        'precio_avg_normal': precio_promedio_normal,
        'sobreprecio_vs_p75': sobreprecio_vs_p75,
        'sobreprecio_vs_promedio': sobreprecio_vs_promedio,
        'cantidad_expensive': len(expensive_data)
    })

# Mostrar resultados
print("TOP 10 COMMODITIES CON MAYOR GASTO EN SOBREPRECIOS")
print("=" * 100)
print(f"Total gastado en sobreprecios: ${total_overspend:,.0f} CLP")
print("=" * 100)

for i, stats in enumerate(commodity_stats, 1):
    percentage_total = (stats['gasto_total'] / total_overspend * 100)
    
    print(f"{i:2d}. {stats['commodity'][:60]:<60}")
    print(f"    Gasto en sobreprecios: ${stats['gasto_total']:,.0f} CLP ({percentage_total:5.1f}%)")
    print(f"    Precio promedio productos caros: ${stats['precio_avg_expensive']:,.0f}")
    print(f"    Precio P75 esperado (referencia): ${stats['precio_p75_esperado']:,.0f}")
    print(f"    Precio promedio normal: ${stats['precio_avg_normal']:,.0f}")
    print(f"    Sobreprecio vs P75: +{stats['sobreprecio_vs_p75']:5.1f}%")
    print(f"    Sobreprecio vs promedio: +{stats['sobreprecio_vs_promedio']:5.1f}%")
    print(f"    Compras caras: {stats['cantidad_expensive']:,}")
    print()

print("=" * 100)

# Estadísticas adicionales
top_10_spending = expensive_spending_by_commodity.head(10).sum()
print(f"Los top 10 commodities concentran: ${top_10_spending:,.0f} CLP ({top_10_spending/total_overspend*100:.1f}%) del gasto total en sobreprecios")

TOP 10 COMMODITIES CON MAYOR GASTO EN SOBREPRECIOS
Total gastado en sobreprecios: $59,495,311,114,801,340,416 CLP
 1. Diesel fuel                                                 
    Gasto en sobreprecios: $5,447,401,414,483,947,520 CLP (  9.2%)
    Precio promedio productos caros: $30,652,141,280,389
    Precio P75 esperado (referencia): $2,917,144
    Precio promedio normal: $6,059,828,752,171
    Sobreprecio vs P75: +1050758313.9%
    Sobreprecio vs promedio: +405.8%
    Compras caras: 412

 2. Sweet biscuits or cookies                                   
    Gasto en sobreprecios: $1,722,703,526,228,867,840 CLP (  2.9%)
    Precio promedio productos caros: $178,897,867,708,636
    Precio P75 esperado (referencia): $2,650
    Precio promedio normal: $19,832,619,865,000
    Sobreprecio vs P75: +6750862932301.4%
    Sobreprecio vs promedio: +802.0%
    Compras caras: 883

 3. Stationery                                                  
    Gasto en sobreprecios: $1,218,611,329,541,026,

In [ ]:
# Crear DataFrame solo con las compras clasificadas como caras
compras_caras = data_merged[data_merged['is_expensive'] == True].copy()

# Ver información básica del DataFrame
print("COMPRAS CLASIFICADAS COMO CARAS")
print("=" * 50)
print(f"Total de compras caras: {len(compras_caras):,}")
print(f"Porcentaje del total: {(len(compras_caras) / len(data_merged) * 100):.2f}%")
print(f"Monto total gastado: ${compras_caras['Monto_Total_Orden'].sum():,.0f} CLP")
print("=" * 50)

# Mostrar las primeras filas
compras_caras.head()

COMPRAS CLASIFICADAS COMO CARAS
Total de compras caras: 61,692
Porcentaje del total: 8.25%
Monto total gastado: $59,495,311,114,801,332,224 CLP


,NombreOC,EstadoOC,RegionUnidadCompra,Proveedor,CodigoProductoONU,NombreItem,DescripcionItem,CantidadItem,MontoTotalItem,TipoCompra,Precio_Unitario_TOTAL,Segment Code,Segment Name,Family Code,Family Name,Class Code,Class Name,Commodity Name,Monto_Total_Orden,is_expensive
15,ORDEN DE COMPRA DESDE 2446-78-LE25,Recepcion Conforme,Region de Coquimbo,SOCIEDAD COMERCIAL M&T SPA,44121710,TIZA PARA ESCRIBIR O ACCESORIOS,TIZA EN POLVO ROJO 1 KILOGRAMO,20,353380,Licitacion,17669.0,44000000.0,Office Equipment and Accessories and Supplies,44120000.0,Office supplies,44121700.0,Writing instruments,Writing chalk or accessories,353380.0,True
36,ALCALDIA. S/P N°367 (CARNAVAL DE LAS MASCARAS ...,Aceptada,Region de Antofagasta,EVENTOS E INVERSIONES SPA,90151602,PRODUCTORAS,CARNAVAL DE MASCARAS 2025. RECADOS A TOCOPILLA,1,15042016,Licitacion,15042016.0,90000000.0,Travel and Food and Lodging and Entertainment ...,90150000.0,Entertainment services,90151600.0,Travelling shows,Touring companies,15042016.0,True
42,CS MANTENCIÓN Y REPARACIÓN VEHÍCULOS PESADOS,Recepcion Conforme,Region del Libertador General Bernardo OHiggins,TALLER MECÁNICO JORGE MORENO E HIJOS LIMITADA,25172503,NEUMÁTICOS PARA CAMIONES PESADOS,Reparación y repuestos de camiones patente SLY...,1,1352319,Licitacion,1352319.0,25000000.0,Commercial and Military and Private Vehicles a...,25170000.0,Transportation components and systems,25172500.0,Tires and tire tubes,Heavy truck tires,1352319.0,True
61,ADQUISICIÓN JARDÍN MIS PRIMEROS PASITOS,Aceptada,Region de los Lagos,FERRETERIAS WEITZLER S A,39121303,CAJAS ELÉCTRICAS,código producto 2-0020-00086 caja d/empalme 7010,1,205714285,Licitacion,205714285.0,39000000.0,Electrical Systems and Lighting and Components...,39120000.0,Electrical equipment and components and supplies,39121300.0,Electrical boxes and enclosures and fittings a...,Electrical boxes,205714285.0,True
82,ADQUISICIÓN DE MATERIALES - DELEGACIÓN DE ROLECHA,Enviada a Proveedor,Region de los Lagos,FERRETERIA EL CONSTRUCTOR LIMITADA,27112802,HOJAS DE SIERRA,Barre hojas truper naranjo,2,15966386,Licitacion,7983193.0,27000000.0,Tools and General Machinery,27110000.0,Hand tools,27112800.0,Tool attachments and accessories,Saw blades,15966386.0,True


In [121]:
# Exportar el DataFrame de compras caras a CSV
compras_caras.to_csv('data/compras_caras.csv', index=False)

print("DataFrame de compras caras exportado a: data/compras_caras.csv")
print(f"Registros exportados: {len(compras_caras):,}")

DataFrame de compras caras exportado a: data/compras_caras.csv
Registros exportados: 61,692
